# Summary statistics on the BioDCASE Task 2 datasets

This notebook computes summary statistics on the [BioDCASE Task 2](https://biodcase.github.io/challenge2026/) datasets as downloaded on [Zenodo](https://zenodo.org/records/18832958)

## Statistics on annotation events

In [45]:
import os
import pandas as pd
from datetime import datetime
import librosa

edition='2026'
base_directory = '/home/datawork-osmose/dataset/BioDCASE/'+edition
csv_name = 'summary_biodcase_'+edition+'.csv'

def count_labels_and_duration_from_csv_file(csv_filepath):
    label_counts = {}
    total_duration = 0.0
    possible_label_cols = ['label', 'event_label', 'class', 'species', 'annotation']
    possible_explicit_duration_cols = ['duration', 'length_seconds']
    start_dt_col_name = 'start_datetime'
    end_dt_col_name = 'end_datetime'

    try:
        df = pd.read_csv(csv_filepath)

        # Process labels (existing logic)
        found_label_col = None
        for col in possible_label_cols:
            if col in df.columns:
                found_label_col = col
                break

        if found_label_col:
            counts = df[found_label_col].value_counts()
            for label, count in counts.items():
                label_counts[label] = label_counts.get(label, 0) + count
        else:
            print(f"Warning: File '{os.path.basename(csv_filepath)}' does not contain any of the expected label columns ({', '.join(possible_label_cols)}). Available columns: {df.columns.tolist()}. Skipping this file's label counting.")

        # Process duration
        found_duration_col = None
        for col in possible_explicit_duration_cols:
            if col in df.columns:
                found_duration_col = col
                break

        if found_duration_col:
            # Use explicit duration column
            total_duration = df[found_duration_col].apply(pd.to_numeric, errors='coerce').fillna(0).sum()
        elif start_dt_col_name in df.columns and end_dt_col_name in df.columns:
            # Calculate duration from start_datetime and end_datetime
            try:
                df[start_dt_col_name] = pd.to_datetime(df[start_dt_col_name])
                df[end_dt_col_name] = pd.to_datetime(df[end_dt_col_name])
                # Calculate duration in seconds and sum them up
                total_duration = (df[end_dt_col_name] - df[start_dt_col_name]).dt.total_seconds().fillna(0).sum()
            except Exception as dt_e:
                print(f"Warning: Error parsing datetime columns in '{os.path.basename(csv_filepath)}': {dt_e}. Skipping duration calculation from datetimes.")
        else:
            print(f"Warning: File '{os.path.basename(csv_filepath)}' does not contain any explicit duration columns ({', '.join(possible_explicit_duration_cols)}) nor both '{start_dt_col_name}' and '{end_dt_col_name}'. Available columns: {df.columns.tolist()}. Skipping this file's duration summing.")

    except FileNotFoundError:
        print(f"Error: File not found at '{csv_filepath}'. Skipping this file.")
    except Exception as e:
        print(f"Error reading file '{os.path.basename(csv_filepath)}': {e}. Skipping this file.")

    return label_counts, total_duration

csv_files = []
folder_type = []

for root, dirs, files in os.walk(base_directory):
    for file in files:
        if file.endswith('.csv'):
            full_path = os.path.join(root, file)
            csv_files.append(full_path)            
            folder_type.append(root.split('/')[-2])
            
all_dataset_data = []

for csv_file_path,folder_type_ind in zip(csv_files,folder_type):
    dataset_name = os.path.splitext(os.path.basename(csv_file_path))[0] # Extract dataset name from filename
    print(f"Processing dataset: {dataset_name}")
    label_counts, total_duration = count_labels_and_duration_from_csv_file(csv_file_path)
    all_dataset_data.append({
        'dataset_name': dataset_name,
        'train_val_eval': folder_type_ind,
        'label_counts': label_counts,
        'total_duration': total_duration
    })

import pandas as pd

# 1. Initialize an empty list named flattened_data.
flattened_data = []

# 2. Iterate through each entry in the all_dataset_data list.
for entry in all_dataset_data:
    dataset_name = entry['dataset_name']
    label_counts = entry['label_counts']
    total_duration = entry['total_duration']
    train_val_eval = entry['train_val_eval']

    # 3. For each entry, create a dictionary that includes the dataset_name,
    #    total_duration, and expands the label_counts dictionary into individual label-count pairs.
    row_data = {'dataset_name': dataset_name,'train_val_eval':train_val_eval, 'total_duration': total_duration}
    row_data.update(label_counts)

    # 4. Append this newly created dictionary to the flattened_data list.
    flattened_data.append(row_data)

# 5. Convert the flattened_data list into a Pandas DataFrame.
#    Store this DataFrame in a variable named combined_data_df.
combined_data_df = pd.DataFrame(flattened_data)

# 6. Fill any NaN values in combined_data_df with 0.
#    Convert float columns (excluding 'total_duration') to int for label counts, as counts are whole numbers.
#    'total_duration' should remain float if it has decimal values.
for col in combined_data_df.columns:
    if col not in ['dataset_name', 'total_duration','train_val_eval']:
        combined_data_df[col] = combined_data_df[col].fillna(0).astype(int)
    elif col == 'total_duration':
        combined_data_df[col] = combined_data_df[col].fillna(0.0)

# Calculate a 'Total Events' column by summing all label counts for each dataset
# Exclude 'dataset_name' and 'total_duration' from the sum
label_columns = [col for col in combined_data_df.columns if col not in ['dataset_name', 'total_duration','train_val_eval']]
combined_data_df['Total_Events'] = combined_data_df[label_columns].sum(axis=1)

Processing dataset: ballenyislands2015
Processing dataset: rosssea2014
Processing dataset: kerguelen2005
Processing dataset: elephantisland2014
Processing dataset: casey2014
Processing dataset: maudrise2014
Processing dataset: elephantisland2013
Processing dataset: greenwich2015
Processing dataset: ddu2021
Processing dataset: kerguelen2020
Processing dataset: casey2017
Processing dataset: kerguelen2015
Processing dataset: kerguelen2014


## Statistics on audio files

In [46]:
dataset_audio_files = {}

# Define common audio file extensions
audio_extensions = ('.wav')

# Iterate through the base directory to find subdirectories (datasets)
for root, dirs, files in os.walk(base_directory):

    parts = root.split(os.sep)
    if 'audio' in parts:
        try:
            # The part after 'audio' should be the dataset name
            audio_index = parts.index('audio')
            dataset_name = parts[audio_index + 1] # e.g., 'ballenyislands2015'

            if dataset_name not in dataset_audio_files:
                dataset_audio_files[dataset_name] = []

            for file in files:
                if file.lower().endswith(audio_extensions):
                    full_file_path = os.path.join(root, file)
                    dataset_audio_files[dataset_name].append(full_file_path)
        except (ValueError, IndexError):
            # Handle cases where 'audio' might be the last part or no part after it
            pass

# After collecting all files, sort them for consistency
for dataset_name in dataset_audio_files:
    dataset_audio_files[dataset_name].sort()

audio_file_durations = {}
for dataset_name, files in dataset_audio_files.items():
    durations = []
    for file_path in files:
        try:
            duration = librosa.get_duration(path=file_path)
            durations.append(duration)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            durations.append(None) # Append None for failed files
    audio_file_durations[dataset_name] = durations

import pandas as pd

df_durations = pd.DataFrame({
    'Dataset': [dataset_name for dataset_name, durations in audio_file_durations.items() for _ in durations],
    'Duration (seconds)': [duration for durations in audio_file_durations.values() for duration in durations]
})

from datetime import datetime

timestamps = []

for dataset_name, files in dataset_audio_files.items():
    for file_path in files:
        try:
            filename = os.path.basename(file_path)
            # Expected format: YYYY-MM-DDTHH-MM-SS_XXX.wav
            # Extract 'YYYY-MM-DDTHH-MM-SS'
            timestamp_part = filename.split('_')[0]

            # Replace '-' in the time part with ':'
            date_time_parts = timestamp_part.split('T')
            date_part = date_time_parts[0]
            time_part = date_time_parts[1]
            corrected_time_part = time_part.replace('-', ':')

            final_timestamp_str = f"{date_part}T{corrected_time_part}"
            timestamps.append(final_timestamp_str)
        except Exception as e:
            print(f"Error extracting timestamp from {file_path}: {e}")
            timestamps.append(None)

# Add the new 'Timestamp' column to the DataFrame
df_durations['Timestamp'] = pd.to_datetime(timestamps)

dataset_intervals = {}

for dataset_name in df_durations['Dataset'].unique():
    # Filter DataFrame for the current dataset
    dataset_df = df_durations[df_durations['Dataset'] == dataset_name].sort_values(by='Timestamp').copy()

    # Calculate the time difference between successive files
    # .diff() computes the difference between current and a previous element
    time_diffs = dataset_df['Timestamp'].diff().dt.total_seconds()

    # Store the calculated time differences
    # The first element will be NaN as there's no preceding file
    df_durations.loc[dataset_df.index, 'Time_Difference_seconds'] = time_diffs

df_durations['duty_cycle_percent'] = round( df_durations['Duration (seconds)'] / df_durations['Time_Difference_seconds'] * 100)

average_duty_cycle = df_durations.groupby('Dataset')['duty_cycle_percent'].mean().reset_index()

import pandas as pd

# Start with the base duty cycle calculation to ensure a clean DataFrame
average_duty_cycle = df_durations.groupby('Dataset')['duty_cycle_percent'].mean().reset_index()

# Calculate number of audio files
num_files = df_durations.groupby('Dataset').size().reset_index(name='Num_Audio_Files')

# Calculate average duration of audio files
avg_duration = df_durations.groupby('Dataset')['Duration (seconds)'].mean().reset_index(name='Avg_Audio_Duration_seconds')

# Calculate total cumulated audio duration
total_duration = df_durations.groupby('Dataset')['Duration (seconds)'].sum().reset_index(name='Total_Audio_Duration_seconds')

# Merge the calculated metrics into the clean average_duty_cycle DataFrame
average_duty_cycle = pd.merge(average_duty_cycle, num_files, on='Dataset', how='left')
average_duty_cycle = pd.merge(average_duty_cycle, avg_duration, on='Dataset', how='left')
average_duty_cycle = pd.merge(average_duty_cycle, total_duration, on='Dataset', how='left')

# Convert Total_Audio_Duration_seconds to hours and round
average_duty_cycle['Total_Audio_Duration_hours'] = (average_duty_cycle['Total_Audio_Duration_seconds'] / 3600).round().astype(int)

# Drop the original 'Total_Audio_Duration_seconds' column
average_duty_cycle = average_duty_cycle.drop(columns=['Total_Audio_Duration_seconds'])

# Round the specified columns and convert to int
average_duty_cycle['duty_cycle_percent'] = average_duty_cycle['duty_cycle_percent'].round().astype(int)
average_duty_cycle['Avg_Audio_Duration_seconds'] = average_duty_cycle['Avg_Audio_Duration_seconds'].round().astype(int)

first_timestamps = df_durations.groupby('Dataset')['Timestamp'].min().reset_index()
first_timestamps.rename(columns={'Timestamp': 'First_Timestamp'}, inplace=True)

end_timestamps = df_durations.groupby('Dataset')['Timestamp'].max().reset_index()
end_timestamps.rename(columns={'Timestamp': 'End_Timestamp'}, inplace=True)

average_duty_cycle = pd.merge(average_duty_cycle, first_timestamps, on='Dataset', how='left')
average_duty_cycle = pd.merge(average_duty_cycle, end_timestamps, on='Dataset', how='left')


In [47]:
# Rename 'Dataset' column in average_duty_cycle to 'dataset_name' for merging consistency
average_duty_cycle_renamed = average_duty_cycle.rename(columns={'Dataset': 'dataset_name'})

# Standardize the case of dataset names before merging
combined_data_df['dataset_name'] = combined_data_df['dataset_name'].str.lower()
average_duty_cycle_renamed['dataset_name'] = average_duty_cycle_renamed['dataset_name'].str.lower()

# Merge the two dataframes on the 'dataset_name' column
final_summary_df = pd.merge(combined_data_df, average_duty_cycle_renamed, on='dataset_name', how='left')

final_summary_df['event_ratio_presence_percent'] = (100000*(final_summary_df['total_duration'] / (final_summary_df['Avg_Audio_Duration_seconds'] * final_summary_df['Num_Audio_Files'])).round(3))/1000


final_summary_df = final_summary_df.rename(columns={'total_duration': 'event_total_duration_seconds','Total_Events': 'event_total_number',
                                                   'First_Timestamp': 'file_first_timestamp', 'End_Timestamp': 'file_last_timestamp',
                                                   'Total_Audio_Duration_hours':'file_total_duration_hours','Num_Audio_Files':'file_total_number',
                                                   'Avg_Audio_Duration_seconds':'file_average_duration','duty_cycle_percent':'file_duty_cycle_percent'})

final_summary_df['event_total_duration_seconds'] = final_summary_df['event_total_duration_seconds'].round().astype(int)

## Format and save final csv files

In [48]:
reorder = ['dataset_name', 'train_val_eval', 
       'file_first_timestamp', 'file_last_timestamp', 'file_total_number','file_total_duration_hours',
       'file_average_duration',  'file_duty_cycle_percent',
       'event_total_duration_seconds', 'event_total_number','event_ratio_presence_percent',  'bp20', 'bma', 'bp20plus', 'bpd', 'bmd',
       'bmb', 'bmz']

final_summary_df=final_summary_df[reorder]
final_summary_df = final_summary_df[final_summary_df['train_val_eval']!='evaluation']

final_summary_df.to_csv(csv_name,index=False)

display(final_summary_df)

,dataset_name,train_val_eval,file_first_timestamp,file_last_timestamp,file_total_number,file_total_duration_hours,file_average_duration,file_duty_cycle_percent,event_total_duration_seconds,event_total_number,event_ratio_presence_percent,bp20,bma,bp20plus,bpd,bmd,bmb,bmz
0,ballenyislands2015,train,2015-01-15 17:00:00,2016-01-10 18:00:00,205,204,3582,2,9992,2222,1.4,951,923,148,78,47,44,31
1,rosssea2014,train,2014-02-09 11:00:00,2014-12-15 08:00:00,176,176,3599,2,531,104,0.1,0,104,0,0,0,0,0
2,kerguelen2005,train,2005-01-31 11:00:00,2006-01-31 14:00:00,200,200,3600,2,12697,2960,1.8,788,812,78,444,435,237,166
3,elephantisland2014,train,2014-01-01 00:00:00,2014-12-31 23:00:00,2592,216,300,8,101289,20964,13.0,4940,6934,2912,4077,1034,967,100
4,casey2014,train,2013-12-25 06:00:00,2014-12-12 07:00:00,194,194,3598,3,51120,6866,7.3,17,3681,0,0,679,1398,1091
5,maudrise2014,train,2014-01-12 11:00:00,2014-09-17 05:00:00,200,83,1500,1,20565,2360,6.9,23,2191,5,6,70,37,28
6,elephantisland2013,train,2013-01-12 13:00:00,2013-12-02 23:00:00,2247,187,300,8,57915,21964,8.6,3662,2625,1859,1042,10840,1785,151
7,greenwich2015,train,2015-01-02 14:09:44,2015-12-31 15:02:29,190,32,602,0,7391,1128,6.5,2,827,1,46,66,157,29
10,casey2017,validation,2016-12-12 20:00:00,2017-11-06 19:00:00,187,187,3598,2,22002,3263,3.3,78,1741,214,0,553,558,119
11,kerguelen2015,validation,2015-02-10 11:00:00,2016-01-27 20:00:00,200,200,3599,2,26610,5542,3.7,552,1970,718,344,1180,542,236
